[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C14/E14-1_Fusion_Laser_Poudre.ipynb)

# Réglage d'un procédé de fabrication additive : prédire la densité d'une pièce

## Le contexte

En fusion laser sur lit de poudre, un faisceau fond la poudre métallique couche après couche. La densité de la pièce dépend de nombreux réglages : puissance du laser, vitesse de balayage, écart entre deux passages, épaisseur de couche, température du plateau, qualité de la poudre et du gaz, position sur le plateau. Une pièce trop poreuse est refusée, mais mesurer chaque éprouvette est long et coûteux. Une campagne de fabrication a relevé 8 000 éprouvettes cubiques, avec leurs réglages et leur densité relative. Le service méthodes voudrait prédire la densité d'un jeu de réglages avant de lancer la fabrication, pour choisir la fenêtre de procédé.

La grandeur à prédire est une densité en pourcentage : c'est un problème de régression.

Défi : prédire `densite_relative_pct` et obtenir un R² supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est le réseau de neurones de scikit-learn du chapitre précédent, avec ses réglages par défaut (une couche de 100 neurones) ; il atteint un R² de 0,85. À vous de faire mieux avec un réseau dense de Keras, dont vous choisirez le nombre d'époques par un arrêt précoce. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Fusion_Laser_Poudre.xlsx`, dans le dossier `donnees` du dépôt : 8 000 éprouvettes, 14 colonnes. Une ligne est une éprouvette cubique fabriquée sur la machine, avec ses réglages et sa densité mesurée.

| Colonne | Signification | Unité |
|---|---|---|
| `alliage` | `316L`, `AlSi10Mg` ou `Ti6Al4V` | catégorie |
| `puissance_laser_W` | puissance du laser | W |
| `vitesse_balayage_mm_s` | vitesse de balayage du faisceau | mm/s |
| `ecart_hachures_mm` | écart entre deux passages voisins | mm |
| `epaisseur_couche_um` | épaisseur d'une couche de poudre | µm |
| `temperature_plateau_C` | température du plateau de fabrication | °C |
| `diametre_poudre_um` | diamètre médian de la poudre | µm |
| `oxygene_ppm` | oxygène résiduel dans la chambre | ppm |
| `position_x_mm`, `position_y_mm` | position de l'éprouvette sur le plateau (le gaz circule vers les y croissants) | mm |
| `debit_gaz_m_s` | vitesse du gaz de balayage | m/s |
| `reutilisations_poudre` | fabrications déjà faites avec la même poudre | cycles |
| `rotation_couche_deg` | rotation de la direction de balayage entre deux couches : 0, 33, 67 ou 90 | degrés |
| `densite_relative_pct` | cible : densité relative de l'éprouvette | % |

Origine : jeu simulé, d'après un bain de fusion de Rosenthal, un critère de recouvrement entre passages, une enthalpie normalisée pour la porosité de keyhole, et l'effet de la poudre, du gaz et des éclaboussures. Quelques mesures d'oxygène et de débit de gaz ne sont pas relevées. Il enseigne une méthode, et aucune conclusion de métallurgie n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la plage de la cible avant toute modélisation : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Fusion_Laser_Poudre.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Fusion_Laser_Poudre.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "densite_relative_pct"
categories = ["alliage"]
mesures = [c for c in df.columns if c not in categories + [cible]]
print("dimensions :", df.shape)
vides = df.isna().sum()
print("cases vides :", vides[vides > 0].to_dict())
print(f"{cible} : de {df[cible].min():.1f} à {df[cible].max():.1f}, médiane {df[cible].median():.1f}, moyenne {df[cible].mean():.1f}")
df.head(3)

Lecture du résultat : 8 000 éprouvettes, 14 colonnes. L'oxygène compte 235 cases vides et le débit de gaz 158, qu'il faudra remplir. La densité va de 77,3 à 100,0 %, pour une médiane de 99,1 % et une moyenne de 97,3 % : la plupart des éprouvettes sont denses, et une minorité est nettement poreuse.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Fusion_Laser_Poudre.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Reproductibilité

Un réseau de neurones part de poids tirés au hasard et lit ses exemples dans un ordre tiré au hasard. La cellule suivante fixe toutes les graines, demande des opérations de calcul déterministes, impose un seul fil de calcul et un jeu d'instructions donné. Avec elle, deux exécutions du notebook sur le même ordinateur donnent les mêmes poids et le même score. Chaque cellule qui construit un réseau rappelle la graine dans sa première ligne : on peut la rejouer seule sans changer le résultat. Sur un autre ordinateur, ou sur un processeur qui mêle deux types de cœurs, les arrondis peuvent différer et le score changer à la troisième décimale.

In [ ]:
# Réseaux de neurones : mêmes résultats à chaque exécution.
# À placer après la cellule de lecture (qui définit GRAINE),
# avant tout usage de TensorFlow.
import os

# Lues à l'import de TensorFlow : journal réduit, oneDNN limité
# au produit matriciel, jeu d'instructions fixé.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["ONEDNN_MAX_CPU_ISA"] = "AVX2"

import numpy as np
import tensorflow as tf
import keras

# Graines, opérations déterministes et un seul fil de calcul.
keras.utils.set_random_seed(GRAINE)
tf.config.experimental.enable_op_determinism()
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| NumPy", np.__version__)

### Étape 2 : mettre le test de côté

Un quart des éprouvettes est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. Dans l'apprentissage restant, un cinquième est réservé à la validation : il servira à décider quand arrêter l'entraînement d'un réseau, jamais le test.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE)
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=GRAINE)
print("apprentissage :", len(X_fit), "éprouvettes")
print("validation :", len(X_val), "éprouvettes")
print("test :", len(X_test), "éprouvettes")

### Étape 3 : regarder la densité sur l'apprentissage

Deux vues de l'apprentissage. À gauche, la distribution de la densité. À droite, la densité contre la densité d'énergie volumique `E = P / (v h t)`, grandeur que les ingénieurs du procédé calculent à partir de la puissance P, de la vitesse v, de l'écart h entre passages et de l'épaisseur t de couche : un symbole par alliage.

In [ ]:
energie = X_train["puissance_laser_W"] / (X_train["vitesse_balayage_mm_s"] * X_train["ecart_hachures_mm"] * X_train["epaisseur_couche_um"] / 1000)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].hist(y_train, bins=40, color="0.6", edgecolor="black", linewidth=0.4)
axes[0].set(xlabel="densité relative (%)", ylabel="éprouvettes")
for alliage, marque, gris in [("316L", "o", "0.75"), ("AlSi10Mg", "s", "0.45"), ("Ti6Al4V", "^", "0.1")]:
    choix = X_train["alliage"] == alliage
    axes[1].scatter(energie[choix], y_train[choix], s=6, marker=marque, color=gris, label=alliage)
axes[1].set(xlabel="densité d'énergie volumique E (J/mm³)", ylabel="densité relative (%)", xscale="log")
axes[1].legend(frameon=False, markerscale=2)
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
dixiemes = pd.qcut(energie, 10, labels=False)
par_dixieme = y_train.groupby(dixiemes).mean()
print(f"densité moyenne : {par_dixieme.iloc[0]:.1f} % au dixième d'énergie le plus bas, {par_dixieme.iloc[4:6].mean():.1f} % au milieu, "
      f"{par_dixieme.iloc[-1]:.1f} % au plus haut")

Lecture du résultat : la densité n'est pas une fonction simple de l'énergie. Elle est la plus basse quand l'énergie est la plus faible (91,6 % en moyenne : le métal fond mal et laisse des vides entre les passages), remonte vers 98,9 % au milieu, puis redescend aux fortes énergies (97,6 % : le bain devient trop creux et piège des pores) : une courbe en cloche, qu'une droite ne suit pas. Elle dépend aussi de l'alliage, et de réglages qui ne figurent pas dans E. Un modèle doit combiner ces effets.

### Étape 4 : construire le modèle de départ

Le modèle de départ est le réseau de neurones de scikit-learn du chapitre précédent, avec ses réglages par défaut : une couche cachée de 100 neurones, 200 époques au plus. Un pipeline prépare d'abord les entrées : l'alliage devient des colonnes de 0 et de 1, les cases vides prennent la médiane de l'apprentissage, chaque mesure est ramenée à une moyenne nulle et à un écart type de 1. Il apprend sur `X_train` seul, la densité telle quelle.

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle prédit les éprouvettes qu'il n'a jamais vues. Trois indicateurs sont lus : le coefficient de détermination R², et deux erreurs en points de densité, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE). Un témoin sans apprentissage, qui répond toujours la moyenne de l'apprentissage, donne le point de comparaison le plus bas.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

y_pred = modele.predict(X_test)
score_depart = r2_score(y_test, y_pred)
score_temoin = r2_score(y_test, np.full(len(y_test), y_train.mean()))
print(f"R² : {score_depart:.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.2f} points de densité")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f} points de densité")
print(f"R² du témoin (toujours la moyenne de l'apprentissage) : {score_temoin:.3f}")
print(f"époques effectuées : {modele[-1].n_iter_} sur {modele[-1].max_iter} permises")

Lecture du résultat : le R² est de 0,85 et l'erreur absolue moyenne de 1,06 point de densité, pour une densité moyenne de 97,1 % sur la base de test. Le témoin, qui répond toujours la moyenne, n'explique rien : son R² est nul, au signe de l'arrondi près. Le réseau a effectué 200 époques sur les 200 permises : il s'est arrêté faute d'époques, sans avoir convergé. C'est ce chiffre, mesuré sur des éprouvettes que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : un réseau dense de Keras et l'arrêt précoce

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les quatre cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première prépare les entrées et la densité réduite (moyenne nulle, écart type 1) pour Keras, avec l'échelle apprise sur la partie d'apprentissage seule. La deuxième construit un réseau dense de trois couches et l'entraîne trois fois : 10 époques, 200 époques sans surveillance, puis jusqu'à 300 époques avec un arrêt précoce, qui suit l'erreur sur la validation, s'arrête quand elle ne baisse plus pendant 20 époques et rend les poids de la meilleure époque. La troisième place sur la même validation le réseau de scikit-learn, avec et sans mise à l'échelle de la cible. La dernière mesure le réseau retenu une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

fig, ax = plt.subplots(figsize=(6.4, 3.6))
historique = histoires["200 époques sans surveillance"].history
ax.plot(range(1, len(historique["loss"]) + 1), historique["loss"], color="0.55", label="apprentissage")
ax.plot(range(1, len(historique["val_loss"]) + 1), historique["val_loss"], color="black", linestyle="--", label="validation")
ax.axvline(meilleure, color="0.3", linewidth=0.8, linestyle=":")
ax.set(xlabel="époque", ylabel="erreur quadratique (densité réduite)", yscale="log")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Votre code :

Lecture du résultat : sur la validation, le réseau de scikit-learn par défaut atteint un R² de 0,68 ; avec la densité mise à l'échelle, le même réseau monte à 0,97 : une ligne de préparation suffit à expliquer l'écart du départ. Avec Keras, 10 époques donnent déjà 0,971, mais 200 époques sans surveillance dépassent le meilleur moment : l'erreur de validation est minimale à l'époque 56 puis remonte, et le R² retombe à 0,944. L'arrêt précoce rend les poids de la meilleure époque et atteint 0,976 en 76 époques. Le réseau retenu (300 époques, arrêt précoce), mesuré une seule fois sur le test, atteint un R² de 0,97, soit 0,12 de plus que le modèle de départ. Remarque : une fois la cible mise à l'échelle, le réseau de scikit-learn (0,968) reste à moins de 0,01 de celui de Keras sur la validation (0,976) ; ce que Keras apporte ici est la liberté de l'architecture et le contrôle de l'entraînement (rappels, taux d'apprentissage, couches de toute forme).

## Le score de référence

Le score publié est le R² du modèle de départ de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, modele.predict(X_test))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.2f" % score)

## Synthèse

1. Un réseau de scikit-learn aux réglages par défaut apprend la densité telle quelle et s'arrête sans avoir convergé : R² de 0,85 sur le test, 0,68 sur la validation.
2. Mettre la cible à l'échelle porte le même réseau à 0,97 sur la validation ; la préparation des entrées ne s'arrête pas aux colonnes d'entrée.
3. Le nombre d'époques se règle sur une validation : trop peu d'époques (0,97 pour 10) sous-entraîne, trop d'époques sans surveillance (0,94 pour 200) laisse le réseau s'ajuster au bruit de l'apprentissage.
4. L'arrêt précoce suit l'erreur de validation, s'arrête après 20 époques sans progrès et rend les poids de la meilleure époque (0,98 sur la validation).
5. La graine est rappelée dans la cellule qui construit le réseau : on la rejoue seule sans changer le résultat. Le test ne sert qu'une fois, au réseau retenu (0,97 ici).